# Capítulo 7: Construir Aplicações de Chat
## Introdução Rápida à API OpenAI

Este caderno é adaptado do [Repositório de Exemplos Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) que inclui cadernos que acedem aos serviços [Azure OpenAI](notebook-azure-openai.ipynb).

A API Python OpenAI funciona também com Modelos Azure OpenAI, com algumas modificações. Saiba mais sobre as diferenças aqui: [Como alternar entre os endpoints OpenAI e Azure OpenAI com Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Visão Geral  
"Grandes modelos de linguagem são funções que mapeiam texto para texto. Dado um texto de entrada, um grande modelo de linguagem tenta prever o texto que se seguirá"(1). Este caderno "quickstart" irá apresentar aos utilizadores conceitos de alto nível sobre LLM, requisitos principais do pacote para começar com AML, uma introdução simples ao design de prompts, e vários exemplos curtos de diferentes casos de uso. 


## Índice  

[Visão Geral](#overview)  
[Como usar o Serviço OpenAI](#how-to-use-openai-service)  
[1. Criar o seu Serviço OpenAI](#1.-creating-your-openai-service)  
[2. Instalação](#2.-installation)    
[3. Credenciais](#3.-credentials)  

[Casos de Uso](#use-cases)    
[1. Resumir Texto](#1.-summarize-text)  
[2. Classificar Texto](#2.-classify-text)  
[3. Gerar Novos Nomes de Produto](#3.-generate-new-product-names)  
[4. Ajustar Finamente um Classificador](#4.fine-tune-a-classifier)  

[Referências](#references)


### Construa o seu primeiro prompt  
Este pequeno exercício fornecerá uma introdução básica para enviar prompts a um modelo da OpenAI para uma tarefa simples de "resumo".


**Passos**:  
1. Instale a biblioteca OpenAI no seu ambiente Python  
2. Carregue as bibliotecas auxiliares padrão e defina as suas credenciais típicas de segurança da OpenAI para o Serviço da OpenAI que criou  
3. Escolha um modelo para a sua tarefa  
4. Crie um prompt simples para o modelo  
5. Envie o seu pedido para a API do modelo!


### 1. Instalar OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importar bibliotecas auxiliares e instanciar credenciais


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Encontrar o modelo certo  
Modelos como o GPT-4o e o GPT-4o mini conseguem compreender e gerar linguagem natural, e estão disponíveis na plataforma OpenAI com diferentes níveis de potência e velocidade adequados a diferentes tarefas.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Design de Prompts  

"A magia dos grandes modelos de linguagem é que ao serem treinados para minimizar este erro de previsão em vastas quantidades de texto, os modelos acabam por aprender conceitos úteis para estas previsões. Por exemplo, aprendem conceitos como"(1):

* como soletrar
* como funciona a gramática
* como parafrasear
* como responder a perguntas
* como manter uma conversa
* como escrever em várias línguas
* como programar
* etc.

#### Como controlar um grande modelo de linguagem  
"De todos os inputs para um grande modelo de linguagem, de longe o mais influente é o texto do prompt(1).

Os grandes modelos de linguagem podem ser incitados a produzir saída de algumas formas:

Instrução: Diga ao modelo o que quer
Completamento: Induza o modelo a completar o início do que quer
Demonstração: Mostre ao modelo o que quer, com:
Alguns exemplos no prompt
Centenas ou milhares de exemplos num conjunto de dados de treino de ajuste fino"



#### Existem três diretrizes básicas para criar prompts:

**Mostrar e dizer**. Torne claro o que quer, quer através de instruções, exemplos, ou uma combinação de ambos. Se quer que o modelo ordene uma lista de itens por ordem alfabética ou que classifique um parágrafo por sentimento, mostre-lhe que é isso que quer.

**Forneça dados de qualidade**. Se está a tentar criar um classificador ou a fazer com que o modelo siga um padrão, certifique-se que há exemplos suficientes. Certifique-se de rever os seus exemplos — o modelo normalmente é inteligente o suficiente para detetar erros ortográficos básicos e dar-lhe uma resposta, mas também pode assumir que isso é intencional e afetar a resposta.

**Verifique as suas definições.** As definições temperature e top_p controlam o quão determinístico é o modelo ao gerar uma resposta. Se está a pedir uma resposta onde só há uma resposta correta, então deve definir estes valores mais baixos. Se pretende respostas mais diversas, talvez queira defini-los mais altos. O erro número um que as pessoas cometem com estas definições é assumir que são controlos de "esperteza" ou "criatividade".


Fonte: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Submeter!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Repetir a mesma chamada, como é que os resultados se comparam?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Resumir Texto  
#### Desafio  
Resumir texto adicionando um 'tl;dr:' no final de um trecho de texto. Repare como o modelo compreende como realizar várias tarefas sem instruções adicionais. Pode experimentar com prompts mais descritivos do que tl;dr para modificar o comportamento do modelo e personalizar o resumo que recebe(3).  

Trabalhos recentes demonstraram ganhos substanciais em muitas tarefas e benchmarks de PLN através do pré-treinamento em um grande corpus de texto seguido de ajuste fino para uma tarefa específica. Embora tipicamente agnóstico à tarefa na arquitetura, este método ainda requer conjuntos de dados de ajuste fino específicos para a tarefa, com milhares ou dezenas de milhares de exemplos. Por outro lado, os humanos geralmente conseguem realizar uma nova tarefa linguística a partir de apenas alguns exemplos ou de instruções simples – algo com que os sistemas de PLN atuais ainda têm bastante dificuldade. Aqui mostramos que aumentar a escala dos modelos de linguagem melhora bastante a performance agnóstica à tarefa com poucos exemplos, por vezes até alcançando competitividade com abordagens anteriores de ajuste fino consideradas de última geração.  



Tl;dr


# Exercícios para vários casos de uso  
1. Resumir Texto  
2. Classificar Texto  
3. Gerar Novos Nomes de Produto


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Classificar Texto  
#### Desafio  
Classifique itens em categorias fornecidas no momento da inferência. No exemplo seguinte, fornecemos tanto as categorias como o texto a classificar no prompt(*playground_reference). 

Pedido do Cliente: Olá, uma das teclas do teclado do meu portátil partiu recentemente e vou precisar de uma substituição:

Categoria classificada:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Gerar Novos Nomes de Produtos
#### Desafio
Criar nomes de produtos a partir de palavras exemplares. Aqui incluímos no prompt informações sobre o produto para o qual vamos gerar nomes. Também fornecemos um exemplo semelhante para mostrar o padrão que desejamos receber. Definimos também o valor de temperatura alto para aumentar a aleatoriedade e respostas mais inovadoras.

Descrição do produto: Uma máquina caseira de milkshake
Palavras-chave: rápido, saudável, compacto.
Nomes de produtos: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descrição do produto: Um par de sapatos que pode servir para qualquer tamanho de pé.
Palavras-chave: adaptável, serve, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referências  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Melhores práticas para afinar modelos GPT para classificar texto](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Para Mais Ajuda  
[Equipa de Comercialização da OpenAI](AzureOpenAITeam@microsoft.com) 


# Contribuidores
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
